# Salary Prediction — Final Report

## Objective
Build and evaluate regression models to predict **Salary** using demographic and professional features.

## Approach
- Load and merge the provided datasets (people, salary, descriptions).
- Establish a baseline using **DummyRegressor**.
- Train a **Linear Regression** model with a clean preprocessing pipeline.
- Run an additional experiment including **Job Title** and compare results.
- Evaluate using **MAE** (and bootstrap confidence intervals for robustness).


In [17]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path().resolve().parent
sys.path.append(str(project_root))


In [18]:
import pandas as pd

from src.data import load_and_merge_data
from src.features import build_preprocessor
from src.models import build_model
from src.evaluation import evaluate_holdout


In [19]:
PEOPLE_PATH_FILE = "../data/raw/people.csv"
SALARY_PATH_FILE = "../data/raw/salary.csv"
DESCRIPTIONS_PATH_FILE = "../data/raw/descriptions.csv"


In [20]:
df = load_and_merge_data(
    people_path=PEOPLE_PATH_FILE,
    salary_path=SALARY_PATH_FILE,
    descriptions_path=DESCRIPTIONS_PATH_FILE
)

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (373, 8)


,id,Age,Gender,Education Level,Job Title,Years of Experience,Salary,Description
0,0,32.0,Male,Bachelor's,Software Engineer,5.0,90000.0,I am a 32-year-old male working as a Software ...
1,1,28.0,Female,Master's,Data Analyst,3.0,65000.0,I am a 28-year-old data analyst with a Master'...
2,2,45.0,Male,PhD,Senior Manager,15.0,150000.0,I am a 45-year-old Senior Manager with a PhD a...
3,3,36.0,Female,Bachelor's,Sales Associate,7.0,60000.0,I am a 36-year-old female Sales Associate with...
4,4,52.0,Male,Master's,Director,20.0,200000.0,I am a 52-year-old male with over two decades ...


In [21]:
df.isna().sum()

id                     0
Age                    3
Gender                 3
Education Level        3
Job Title              3
Years of Experience    0
Salary                 0
Description            3
dtype: int64

In [22]:
pre_base = build_preprocessor(include_job_title=False)
pipe_dummy = build_model("dummy", pre_base)

res_dummy = evaluate_holdout(df, target_col="Salary", pipeline=pipe_dummy, with_ci=True)
res_dummy


{'mae': 40533.333333333336,
 'n_train': 298,
 'n_test': 75,
 'mae_boot_mean': 40473.13333333333,
 'mae_ci_low': 34463.33333333333,
 'mae_ci_high': 47133.333333333336}

In [23]:
pipe_linear_base = build_model("linear", pre_base)

res_linear_base = evaluate_holdout(df, target_col="Salary", pipeline=pipe_linear_base, with_ci=True)
res_linear_base


{'mae': 10797.999241268952,
 'n_train': 298,
 'n_test': 75,
 'mae_boot_mean': 10798.169699093474,
 'mae_ci_low': 8539.53484842004,
 'mae_ci_high': 13626.336524585484}

In [24]:
pre_job = build_preprocessor(include_job_title=True)
pipe_linear_job = build_model("linear", pre_job)

res_linear_job = evaluate_holdout(df, target_col="Salary", pipeline=pipe_linear_job, with_ci=True)
res_linear_job


{'mae': 11371.572373301655,
 'n_train': 298,
 'n_test': 75,
 'mae_boot_mean': 11354.087479701811,
 'mae_ci_low': 8769.990362862456,
 'mae_ci_high': 14839.644655977329}

In [25]:
results = pd.DataFrame([
    {"model": "Dummy (baseline)", **res_dummy},
    {"model": "Linear (no Job Title)", **res_linear_base},
    {"model": "Linear (+ Job Title)", **res_linear_job},
])

cols_order = ["model", "mae", "mae_ci_low", "mae_ci_high", "n_train", "n_test"]
results[cols_order].sort_values("mae")


,model,mae,mae_ci_low,mae_ci_high,n_train,n_test
1,Linear (no Job Title),10797.999241,8539.534848,13626.336525,298,75
2,Linear (+ Job Title),11371.572373,8769.990363,14839.644656,298,75
0,Dummy (baseline),40533.333333,34463.333333,47133.333333,298,75


## Conclusions

- The **DummyRegressor** provides a baseline error level. Any useful model must improve on it.
- The **Linear Regression** model significantly improves MAE compared to the baseline.
- Adding **Job Title** is tested as an additional experiment:
  - Adding **Job Title** was tested as an additional experiment, but it **did not improve MAE** on the holdout set.
  - Given its high cardinality and the lack of performance gain, it was **excluded** from the final model to favor robustness/generalization.

